# Feature discovery

The next goal is to discover our features. We want the representation that carries the most information about solubility with the fewest features, so several representations are compared under one model and one protocol, and the chosen set is carried forward.

Upstream, [01_prep_and_analysis.ipynb](01_prep_and_analysis.ipynb) hands over one vote per independent measurement, with gross errors flagged; [02_tiers.ipynb](02_tiers.ipynb) assigns every vote one of four tiers and measures how well a second vote predicts the first; [03_splits.ipynb](03_splits.ipynb) provides random and cluster 10-fold splits grouped by skeleton.

## Framework

- **Data.** One row per vote with its tier and weight $1/n$, where $n$ is the compound's number of unflagged votes, so every compound counts once. Flagged votes carry weight zero and are set aside. A compound whose votes fall in different tiers keeps more than one: the labels keep the disagreement the floor was estimated with.
- **Splits.** The random cross-validation splits select representations: split $k$ trains on eight folds, chooses on fold $k+1$ (validation) and audits on fold $k$ (test), which plays no part in any choice. The chosen sets are then also scored on the cluster splits, where a test compound has no close relative in training.
- **Model.** A random forest classifier over the four tiers, with the same settings for every feature set so differences come from the features. The settings are our choice and are not tuned. Every score is therefore what *this forest extracts* from a representation — a lower bound on what the representation holds, not a measurement of it.
- **Scores.** The forest outputs a probability for each tier. Its **information** about the measured tier is $H(T) - \text{log-loss}$: the bits of uncertainty about the tier that the prediction removes, where the log-loss is the vote-weighted average of $-\log_2$ of the probability given to the tier actually measured. This counts a confident wrong answer against a model and a hedged right one for it, which a hard predicted tier cannot. Probabilities are first shrunk toward uniform, $q' = (1-\lambda)\,q + \lambda/K$, with $\lambda$ chosen on the validation side of each split from a grid whose floor is 0.005: a one-parameter calibration, so a forest's over-confidence is not mistaken for a representation's weakness, and no probability can be exactly zero — a forest can give a tier it never saw for such inputs a probability of exactly zero, and one such vote would otherwise cost an infinite log-loss. The hard predicted tier (the most probable one) gives the accuracies shown alongside: same tier, one tier off, two or more off.
- **Populations.** Two are scored, each labelled with its number of compounds; over the 10 splits every compound is held out once. **All compounds** is the headline and the one feature sets are chosen on. **Compounds with 2+ unflagged votes** are the only ones on which the repeat-measurement benchmark exists, so the model is compared with the benchmark on them.
- **Uncertainty.** Mean over the $J = 10$ splits with a standard error corrected for the overlap between training sets, $\sqrt{(1/J + n_\text{test}/n_\text{train})\, s^2}$, where $s^2$ is the variance of the split scores.
- **Reference points.** The tier's entropy, about 1.98 bits over all compounds: the information is also shown as a share of it, but no model can reach it, because the measured tier is itself noisy. The **repeat-measurement benchmark** from the tiers notebook, held out on compounds with 2+ unflagged votes: a second vote shares 0.963 bits with the first and lands in the same tier 76.5% of the time. A model that matched it would be as good as measuring the compound again — but it is not a ceiling either: a second measurement is a noisy witness, and a perfect model of a compound's true solubility could share more with a measured tier than another measurement does. The reachable ceiling lies between the benchmark and the tier's entropy.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdCIPLabeler, rdFingerprintGenerator
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestClassifier

RDLogger.DisableLog("rdApp.warning")  # AqSolDB's [H+]/hydride fragments warn on every parse

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

votes = pd.read_csv(ROOT / "data" / "aqsoldb_tiers.csv")  # one row per vote: tier, Gray-coded bits, weight
values = votes[votes["weight"] > 0].reset_index(drop=True)  # flagged votes carry weight zero: set aside
TIERS = int(values["tier"].max()) + 1
splits = pd.read_csv(ROOT / "data" / "aqsoldb_splits.csv", index_col="InChIKey")  # grouped 10-fold splits
CV_SPLITS = [c for c in splits.columns if c.startswith("cv_") and c[3:].isdigit()]
CLUSTER_SPLITS = [c for c in splits.columns if c.startswith("cluster_") and c[8:].isdigit()]  # not `cluster_fold`
repeated = (values.groupby("InChIKey")["InChIKey"].transform("size") > 1).to_numpy()  # rows of compounds with 2+ votes

# Features are computed once per compound, from its original SMILES (counterions included).
mols = values.drop_duplicates("InChIKey").set_index("InChIKey")["SMILES"].map(Chem.MolFromSmiles)
for m in mols:
    rdCIPLabeler.AssignCIPLabels(m)  # R/S on stereocentres, for the chirality features
print(
    f"{len(values)} votes ({len(votes) - len(values)} flagged set aside), {len(mols)} compounds, {TIERS} tiers, "
    f"{len(CV_SPLITS)} random and {len(CLUSTER_SPLITS)} cluster splits"
)

12013 votes (84 flagged set aside), 9946 compounds, 4 tiers, 10 random and 10 cluster splits


## Feature sets

1. **ESOL** (Delaney 2004): logP, molecular weight, rotatable bonds and aromatic proportion, using RDKit's equivalents (Crippen logP in place of Daylight's).
2. **AqSolDB** (Sorkun et al. 2019): the 17 RDKit descriptors published with the dataset.
3. **RDKit descriptors**: all of them after dropping near-constant ones, and reduced to one descriptor per group of correlated descriptors. We found no published decorrelated list for RDKit, and correlation depends on the dataset, so we group on our own compounds: every pair in a group has $|\rho_\text{Spearman}|$ at least the chosen level, and the member most correlated with the rest stands for the group. The levels are our choice. The grouping never sees solubility, so running it on all compounds does not leak into the test folds.
4. **Graph environments**: a fixed, unlearned graph representation. Each atom is labelled by atomic number, formal charge, hybridization and chirality; each bond by its order (aromatic included). An atom's label is repeatedly combined with its neighbours' (bond order, label) pairs, which is the Morgan algorithm. We count each distinct environment up to radius 0, 1, 2 and 3 without folding: RDKit hashes each environment to a 32-bit identifier, and we keep one column per identifier, holding how often it occurs, instead of folding identifiers into a fixed-length fingerprint where unrelated environments share a bit. RDKit by default skips an environment that covers the same bonds as one already counted; we keep them, so every bonded atom contributes one environment at each radius (an isolated ion, only at radius 0), even where the molecule ends before that radius. So every feature is a readable substructure. Two environments could hash to the same identifier; the code prints how many such collisions to expect, which is well under one. Environments found in fewer than 10 compounds are dropped; that cutoff is our choice.
5. **Atom and bond feature sums**: Chemprop's atom features (element, degree, formal charge, chirality, hydrogen count, hybridization, aromaticity, mass) and bond features (order, conjugation, ring membership, stereo), summed over the molecule. There is no message passing, so this records composition but not which atoms are bonded to which.
6. **Fixed message passing**: Chemprop's directed-bond message passing over the same atom and bond features, with weights drawn at random and never trained, summed over atoms. Depth 1, 2 and 3 reach that many bonds out, like the environments' radius. Each message passes through a nonlinearity before it is summed, so the result depends on which atoms are bonded to which. The 300 outputs are random projections, so they are not individually readable. Another seed gives a different but equally arbitrary projection, so we repeat this with five seeds (our choice) and look at how much the scores move between them.

Chirality is the CIP label (R or S). RDKit's raw chiral tag (clockwise or anticlockwise) depends on the order of atoms in the SMILES, so the same compound written two ways would get different labels. Note that the descriptor sets (1–3) are blind to stereochemistry altogether; the graph sets (4–6) are not.

In [2]:
AQSOLDB_17 = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors", "NumHeteroatoms", "NumRotatableBonds",
    "NumValenceElectrons", "NumAromaticRings", "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA",
    "LabuteASA", "BalabanJ", "BertzCT",
]  # the descriptor columns published with AqSolDB's curated set

# All RDKit descriptors. Infinite values become missing (the forest handles missing values); huge finite values are
# clipped, because scikit-learn's trees work in float32.
descriptors = pd.DataFrame([Descriptors.CalcMolDescriptors(m) for m in mols], index=mols.index)
descriptors = descriptors.replace([np.inf, -np.inf], np.nan).clip(-1e30, 1e30)
print(f"{descriptors.shape[1]} RDKit descriptors; {descriptors.isna().any(axis=1).sum()} compounds have at least one missing")

esol = pd.DataFrame(
    {
        "MolLogP": descriptors["MolLogP"],
        "MolWt": descriptors["MolWt"],
        "NumRotatableBonds": descriptors["NumRotatableBonds"],
        "AromaticProportion": mols.map(lambda m: sum(a.GetIsAromatic() for a in m.GetAtoms()) / m.GetNumHeavyAtoms()),
    }
)
FEATURE_SETS = {"ESOL (4)": esol, "AqSolDB (17)": descriptors[AQSOLDB_17]}

217 RDKit descriptors; 884 compounds have at least one missing


In [3]:
NEAR_CONSTANT = 0.99  # drop a descriptor when one value covers more than this share of compounds; our choice
LEVELS = [0.95, 0.9, 0.8]  # |Spearman rho| at which descriptors count as one group; our choice

top_share = descriptors.apply(lambda c: c.value_counts(normalize=True, dropna=False).iloc[0])
varying = descriptors.loc[:, top_share <= NEAR_CONSTANT]


def correlation_groups(features: pd.DataFrame, level: float, linkage: str = "complete") -> dict[str, list[str]]:
    """Groups of columns by |Spearman rho|, keyed by the member most correlated with the rest.

    Complete linkage: every pair within a group reaches `level`. Single linkage: every pair that reaches `level` is in
    one group, joined directly or through a chain of such pairs.
    """
    rho = features.corr(method="spearman").abs().fillna(0)  # pairs with no defined correlation count as unrelated
    tree = hierarchy.linkage(squareform(1 - rho.to_numpy(), checks=False), method=linkage)
    labels = pd.Series(hierarchy.fcluster(tree, t=1 - level, criterion="distance"), index=rho.index)
    return {rho.loc[cols, cols].mean().idxmax(): list(cols) for cols in labels.groupby(labels).groups.values()}


FEATURE_SETS[f"RDKit, all ({varying.shape[1]})"] = varying
GROUPS = {}  # feature set -> {descriptor: the correlated descriptors it stands for}, used to read importances
for level in LEVELS:
    groups = correlation_groups(varying, level)
    name = f"RDKit, one per group at |ρ| ≥ {level} ({len(groups)})"
    FEATURE_SETS[name], GROUPS[name] = varying[list(groups)], groups
print(f"{descriptors.shape[1] - varying.shape[1]} near-constant descriptors dropped")

30 near-constant descriptors dropped


In [4]:
RADII = [0, 1, 2, 3]
MIN_COMPOUNDS = 10  # drop environments found in fewer compounds; our choice
# Keep environments RDKit skips by default as repeats of ones already counted, so counts are exact.
generators = {r: rdFingerprintGenerator.GetMorganGenerator(radius=r, includeRedundantEnvironments=True) for r in RADII}
CIP = {"R": 1, "S": 2}  # chirality; 0 = none


def atom_label(a: Chem.Atom) -> int:
    """(atomic number, formal charge, hybridization, chirality) packed into one integer."""
    charge = a.GetFormalCharge()
    assert -16 < charge < 16, f"formal charge {charge} does not fit the label"
    chirality = CIP.get(a.GetProp("_CIPCode"), 0) if a.HasProp("_CIPCode") else 0
    return ((a.GetAtomicNum() * 32 + charge + 16) * 16 + int(a.GetHybridization())) * 4 + chirality


def environments(m: Chem.Mol, radius: int) -> tuple[dict, dict]:
    """Count of each atom environment up to `radius` bonds, and one (atom, radius) where each occurs.

    Atom label = atom_label(); bond label = bond order, aromatic included.
    """
    labels = [atom_label(a) for a in m.GetAtoms()]
    info = rdFingerprintGenerator.AdditionalOutput()
    info.AllocateBitInfoMap()
    counts = generators[radius].GetSparseCountFingerprint(m, customAtomInvariants=labels, additionalOutput=info)
    return counts.GetNonzeroElements(), {env: where[0] for env, where in info.GetBitInfoMap().items()}


def describe_environment(m: Chem.Mol, atom: int, radius: int) -> str:
    """SMILES of the atoms and bonds within `radius` bonds of `atom`, plus the radius and the centre atom's full label.

    SMILES alone hides hybridization, so e.g. sp3 and sp2 carbon would both read "C". Near the edge of a molecule the
    environment reaches fewer than `radius` bonds, so environments of different radius can share a SMILES.
    """
    bonds = list(Chem.FindAtomEnvironmentOfRadiusN(m, radius, atom, enforceSize=False)) if radius else []
    ends = {i for b in bonds for i in (m.GetBondWithIdx(b).GetBeginAtomIdx(), m.GetBondWithIdx(b).GetEndAtomIdx())}
    smiles = Chem.MolFragmentToSmiles(m, atomsToUse=sorted({atom} | ends), bondsToUse=bonds or None)
    a = m.GetAtomWithIdx(atom)
    chirality = f" {a.GetProp('_CIPCode')}" if a.HasProp("_CIPCode") else ""
    return f"{smiles} [radius {radius}, centre {a.GetSymbol()} {a.GetHybridization()} charge {a.GetFormalCharge():+d}{chirality}]"


ENVIRONMENTS = {}  # environment column -> readable description, used to read importances
for r in RADII:
    found = [environments(m, r) for m in mols]
    keep = {env for env, n in Counter(env for counts, _ in found for env in counts).items() if n >= MIN_COMPOUNDS}
    table = pd.DataFrame([{f"env_{e}": c for e, c in counts.items() if e in keep} for counts, _ in found], index=mols.index)
    FEATURE_SETS[f"graph, radius {r} ({len(keep)})"] = table.fillna(0).astype(np.float32)
    for m, (_, where) in zip(mols, found):  # describe each kept environment by the first compound that has it
        for env, (atom, radius) in where.items():
            if env in keep and f"env_{env}" not in ENVIRONMENTS:
                ENVIRONMENTS[f"env_{env}"] = describe_environment(m, atom, radius)
n_ids = len({env for counts, _ in found for env in counts})  # `found` now holds radius 3, which includes radii 0-2
print(f"{n_ids} distinct environments up to radius {RADII[-1]}; expected 32-bit identifier collisions {n_ids**2 / 2**33:.2f}")

56578 distinct environments up to radius 3; expected 32-bit identifier collisions 0.37


In [5]:
from rdkit.Chem.rdchem import BondStereo, BondType, HybridizationType as Hyb

# Chemprop's atom and bond features. Each one-hot ends with an "other" slot for values outside its list; no stereo has
# its own slot ("none", STEREONONE), as in Chemprop. Chirality is CIP R/S (see above); pseudo-asymmetric r/s count as other.
ATOM_ONE_HOT = {
    "element": ([Chem.GetPeriodicTable().GetElementSymbol(z) for z in range(1, 101)], lambda a: a.GetSymbol()),
    "degree": (list(range(6)), lambda a: a.GetDegree()),
    "charge": ([-2, -1, 0, 1, 2], lambda a: a.GetFormalCharge()),
    "chirality": (["none", "R", "S"], lambda a: a.GetProp("_CIPCode") if a.HasProp("_CIPCode") else "none"),
    "H count": (list(range(5)), lambda a: a.GetTotalNumHs()),
    "hybridization": ([Hyb.SP, Hyb.SP2, Hyb.SP3, Hyb.SP3D, Hyb.SP3D2], lambda a: a.GetHybridization()),
}
ATOM_SCALARS = {"aromatic": lambda a: a.GetIsAromatic(), "mass / 100": lambda a: a.GetMass() / 100}
BOND_ONE_HOT = {
    "order": ([BondType.SINGLE, BondType.DOUBLE, BondType.TRIPLE, BondType.AROMATIC], lambda b: b.GetBondType()),
    "stereo": (
        [BondStereo.STEREONONE, BondStereo.STEREOZ, BondStereo.STEREOE, BondStereo.STEREOCIS, BondStereo.STEREOTRANS],
        lambda b: b.GetStereo(),
    ),
}
BOND_SCALARS = {"conjugated": lambda b: b.GetIsConjugated(), "in ring": lambda b: b.IsInRing()}


def names(kind: str, one_hot: dict, scalars: dict) -> list[str]:
    return [f"{kind} {k}={c}" for k, (choices, _) in one_hot.items() for c in [*map(str, choices), "other"]] + [f"{kind} {k}" for k in scalars]


def featurize(item, one_hot: dict, scalars: dict) -> list[float]:
    out = []
    for choices, get in one_hot.values():
        v = get(item)
        out += [float(v == c) for c in choices] + [float(v not in choices)]
    return out + [float(get(item)) for get in scalars.values()]


ATOM_NAMES, BOND_NAMES = names("atom", ATOM_ONE_HOT, ATOM_SCALARS), names("bond", BOND_ONE_HOT, BOND_SCALARS)
HIDDEN, DEPTHS, MP_SEEDS = 300, [1, 2, 3], range(5)  # Chemprop's default hidden size; weights are fixed, never trained
d_atom, d_bond = len(ATOM_NAMES), len(BOND_NAMES)
relu = lambda z: np.maximum(z, 0)


def draw_weights(seed: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Random input, message and output weights, scaled by fan-in."""
    draw = np.random.default_rng(seed)
    return (
        draw.normal(0, 1 / np.sqrt(d_atom + d_bond), (d_atom + d_bond, HIDDEN)),
        draw.normal(0, 1 / np.sqrt(HIDDEN), (HIDDEN, HIDDEN)),
        draw.normal(0, 1 / np.sqrt(d_atom + HIDDEN), (d_atom + HIDDEN, HIDDEN)),
    )


MP_WEIGHTS = {seed: draw_weights(seed) for seed in MP_SEEDS}


def message_passing(X: np.ndarray, src: np.ndarray, dst: np.ndarray, E: np.ndarray, weights: tuple, max_depth: int) -> list[np.ndarray]:
    """Chemprop's directed-bond message passing with fixed weights, summed over atoms after each depth 1..max_depth.

    The message on bond v->w carries what reached v from every neighbour except w; depth counts bond hops.
    """
    W_in, W_msg, W_out = weights
    z0 = np.hstack([X[src], E]) @ W_in  # Chemprop's H_0: added back, before the nonlinearity, at every depth
    h, reverse = relu(z0), np.arange(len(src)) ^ 1  # directed bonds come in pairs: i->j, j->i
    readouts = []
    for depth in range(1, max_depth + 1):
        if depth > 1:
            h = relu(z0 + (arriving[src] - h[reverse]) @ W_msg)
        arriving = np.zeros((len(X), HIDDEN))
        np.add.at(arriving, dst, h)
        readouts.append(relu(np.hstack([X, arriving]) @ W_out).sum(axis=0))
    return readouts


sums, passed = [], {(seed, d): [] for seed in MP_SEEDS for d in DEPTHS}
for m in mols:
    X = np.array([featurize(a, ATOM_ONE_HOT, ATOM_SCALARS) for a in m.GetAtoms()])
    B = np.array([featurize(b, BOND_ONE_HOT, BOND_SCALARS) for b in m.GetBonds()]).reshape(-1, d_bond)
    ends = np.array([(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()], dtype=int).reshape(-1, 2)
    src, dst = ends.ravel(), ends[:, ::-1].ravel()  # each bond in both directions
    sums.append(np.concatenate([X.sum(axis=0), B.sum(axis=0)]))
    for seed in MP_SEEDS:
        for d, readout in zip(DEPTHS, message_passing(X, src, dst, np.repeat(B, 2, axis=0), MP_WEIGHTS[seed], max(DEPTHS))):
            passed[seed, d].append(readout)

FEATURE_SETS[f"atom + bond feature sums ({d_atom + d_bond})"] = pd.DataFrame(sums, index=mols.index, columns=ATOM_NAMES + BOND_NAMES).astype(np.float32)
for (seed, d), readouts in passed.items():
    FEATURE_SETS[f"fixed message passing, depth {d}, seed {seed} ({HIDDEN})"] = pd.DataFrame(
        readouts, index=mols.index, columns=[f"mp_{i}" for i in range(HIDDEN)]
    ).astype(np.float32)
pd.Series({name: features.shape[1] for name, features in FEATURE_SETS.items()}, name="features")

ESOL (4)                                           4
AqSolDB (17)                                      17
RDKit, all (187)                                 187
RDKit, one per group at |ρ| ≥ 0.95 (165)         165
RDKit, one per group at |ρ| ≥ 0.9 (152)          152
RDKit, one per group at |ρ| ≥ 0.8 (131)          131
graph, radius 0 (51)                              51
graph, radius 1 (494)                            494
graph, radius 2 (2107)                          2107
graph, radius 3 (4069)                          4069
atom + bond feature sums (145)                   145
fixed message passing, depth 1, seed 0 (300)     300
fixed message passing, depth 2, seed 0 (300)     300
fixed message passing, depth 3, seed 0 (300)     300
fixed message passing, depth 1, seed 1 (300)     300
fixed message passing, depth 2, seed 1 (300)     300
fixed message passing, depth 3, seed 1 (300)     300
fixed message passing, depth 1, seed 2 (300)     300
fixed message passing, depth 2, seed 2 (300)  

## Random forest across the 10 splits

Each split's forest is trained on its training rows, calibrated on its validation rows ($\lambda$), and scored on both held-out sides; only the validation scores are shown here, for every feature set. Each compound counts once, and every value is the mean over the 10 splits ± the standard error corrected for overlapping training sets.

- **information (bits)**: $H(T) - \text{log-loss}$ of the calibrated predicted distribution — the uncertainty about the measured tier that the prediction removes.
- **share of tier entropy (%)**: that information as a share of $H(T)$; a reference, not a reachable ceiling.
- **accuracy: same tier (%)**, **1 tier off (%)**, **2+ tiers off (%)**: where the most probable tier lands relative to the measured one.

Rows are sorted by information over all compounds. The five seeds of each fixed message-passing depth are averaged per split, and the next table shows how far the seeds spread. The repeat-measurement benchmark has a row for the population it exists on.

In [6]:
import re

RF_SETTINGS = dict(n_estimators=500, n_jobs=-1, random_state=0)  # the same for every feature set; not tuned
LAMBDAS = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]  # shrinkage toward uniform tried on validation; our grid, floor > 0
APART = np.abs(np.subtract.outer(range(TIERS), range(TIERS)))  # tiers between measured and predicted
ALL = f"all compounds ({len(mols):,})"
REPEATED = f"compounds with 2+ unflagged votes ({values.loc[repeated, 'InChIKey'].nunique():,})"
SUBSETS = {ALL: np.ones(len(values), bool), REPEATED: repeated}
METRICS = {  # decimals shown
    "information (bits)": 3,
    "share of tier entropy (%)": 1,
    "accuracy: same tier (%)": 1,
    "1 tier off (%)": 1,
    "2+ tiers off (%)": 1,
}
BENCHMARK = {"information (bits)": 0.963, "accuracy: same tier (%)": 76.5}  # tiers notebook, held out, 2+-vote compounds


def entropy(p: np.ndarray) -> float:
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def information(measured: np.ndarray, proba: np.ndarray, w: np.ndarray, lam: float) -> tuple[float, float]:
    """H(T) - log-loss of the shrunk probabilities, and H(T), both in bits, vote-weighted."""
    q = (1 - lam) * proba + lam / TIERS
    h = entropy(np.bincount(measured, weights=w, minlength=TIERS) / w.sum())
    with np.errstate(divide="ignore"):
        log_loss = -(w * np.log2(q[np.arange(len(measured)), measured])).sum() / w.sum()
    return h - log_loss, h


def calibrate(measured: np.ndarray, proba: np.ndarray, w: np.ndarray) -> float:
    """The shrinkage toward uniform with the lowest log-loss on these (validation) rows."""
    return max(LAMBDAS, key=lambda lam: information(measured, proba, w, lam)[0])


def score(measured: np.ndarray, proba: np.ndarray, w: np.ndarray, lam: float) -> dict:
    """Information of the calibrated prediction, plus where the most probable tier lands."""
    info, h = information(measured, proba, w, lam)
    p = np.zeros((TIERS, TIERS))
    np.add.at(p, (measured, proba.argmax(axis=1)), w)
    p /= p.sum()
    return {
        "information (bits)": info,
        "share of tier entropy (%)": 100 * info / h,
        "accuracy: same tier (%)": 100 * p[APART == 0].sum(),
        "1 tier off (%)": 100 * p[APART == 1].sum(),
        "2+ tiers off (%)": 100 * p[APART >= 2].sum(),
    }


def fit(name: str, split: str):
    """Forest trained on `split`'s training rows; returns a function giving (measured, probabilities, weight, mask) per side."""
    X = FEATURE_SETS[name].reindex(values["InChIKey"]).reset_index(drop=True)  # one row per vote
    y, w = values["tier"].to_numpy(), values["weight"].to_numpy()
    where = values["InChIKey"].map(splits[split]).to_numpy()
    train = where == "train"
    assert train.any() and (where == "validation").any() and (where == "test").any(), f"{split}: a side has no rows"
    forest = RandomForestClassifier(**RF_SETTINGS).fit(X[train], y[train], sample_weight=w[train])
    assert list(forest.classes_) == list(range(TIERS)), "a tier is missing from the training rows"

    def held_out(side: str):
        m = where == side
        return y[m], forest.predict_proba(X[m]), w[m], m

    return held_out


def run(name: str, split_list: list[str]) -> pd.DataFrame:
    """Scores of `name` on the validation and test rows of every split in `split_list`; λ comes from validation."""
    rows = []
    for split in split_list:
        held_out = fit(name, split)
        y_v, q_v, w_v, _ = held_out("validation")
        lam = calibrate(y_v, q_v, w_v)
        for side in ["validation", "test"]:
            y_s, q_s, w_s, m_s = held_out(side)
            for subset, mask in SUBSETS.items():
                k = mask[m_s]
                rows.append({"feature set": name, "split": split, "side": side, "λ": lam, "subset": subset, **score(y_s[k], q_s[k], w_s[k], lam)})
    return pd.DataFrame(rows)


def family(name: str) -> str:
    """Feature set name, with the seeds of a fixed message-passing depth pooled into one."""
    return re.sub(r", seed \d+", "", name)


def members(fam: str) -> list[str]:
    """Feature sets in a family: itself, or the five seeds of a message-passing depth."""
    return [name for name in FEATURE_SETS if family(name) == fam]


def formatted(df: pd.DataFrame) -> pd.DataFrame:
    """Each metric as text, with its own number of decimals."""
    return pd.DataFrame({m: df[m].map(f"{{:.{d}f}}".format) for m, d in METRICS.items()}, index=df.index)


def test_to_train(split_list: list[str]) -> float:
    return (splits[split_list[0]] == "test").sum() / (splits[split_list[0]] == "train").sum()


def summarize(results: pd.DataFrame, side: str, split_list: list[str]) -> pd.DataFrame:
    """Mean over splits ± standard error corrected for overlapping training sets, for one side of one split scheme.

    The seeds of a fixed message-passing depth are averaged per split first. Rows are sorted by information over all compounds.
    """
    on = results[results["side"] == side]
    pooled = on.assign(**{"feature set": on["feature set"].map(family)})
    by = pooled.groupby(["feature set", "subset", "split"], sort=False)[list(METRICS)].mean().groupby(level=[0, 1], sort=False)
    mean, se = by.mean(), np.sqrt((1 / len(split_list) + test_to_train(split_list)) * by.var())
    order = mean.xs(ALL, level="subset")["information (bits)"].sort_values(ascending=False).index
    table = (formatted(mean) + " ± " + formatted(se)).reindex(order, level="feature set")
    table.loc[("repeat measurement (benchmark)", REPEATED), :] = [f"{BENCHMARK[m]:.{d}f}" if m in BENCHMARK else "" for m, d in METRICS.items()]
    return table

In [7]:
results = pd.concat([run(name, CV_SPLITS) for name in FEATURE_SETS], ignore_index=True)
summarize(results, "validation", CV_SPLITS)

information (bits)  \
feature set                              subset                                                         
RDKit, all (187)                         all compounds (9,946)                          1.009 ± 0.010   
                                         compounds with 2+ unflagged votes (1,580)      0.994 ± 0.041   
RDKit, one per group at |ρ| ≥ 0.95 (165) all compounds (9,946)                          1.009 ± 0.010   
                                         compounds with 2+ unflagged votes (1,580)      0.992 ± 0.040   
RDKit, one per group at |ρ| ≥ 0.9 (152)  all compounds (9,946)                          1.009 ± 0.009   
                                         compounds with 2+ unflagged votes (1,580)      0.995 ± 0.043   
RDKit, one per group at |ρ| ≥ 0.8 (131)  all compounds (9,946)                          1.006 ± 0.010   
                                         compounds with 2+ unflagged votes (1,580)      0.994 ± 0.041   
AqSolDB (17)                             all compounds (9,946)                          0.947 ± 0.010   
                                         compounds with 2+ unflagged votes (1,580)      0.928 ± 0.044   
graph, radius 1 (494)                    all compounds (9,946)                          0.899 ± 0.014   
                                         compounds with 2+ unflagged votes (1,580)      0.823 ± 0.046   
atom + bond feature sums (145)           all compounds (9,946)                          0.877 ± 0.014   
                                         compounds with 2+ unflagged votes (1,580)      0.839 ± 0.046   
fixed message passing, depth 1 (300)     all compounds (9,946)                          0.871 ± 0.010   
                                         compounds with 2+ unflagged votes (1,580)      0.847 ± 0.039   
graph, radius 2 (2107)                   all compounds (9,946)                          0.867 ± 0.016   
                                         compounds with 2+ unflagged votes (1,580)      0.797 ± 0.040   
graph, radius 3 (4069)                   all compounds (9,946)                          0.846 ± 0.018   
                                         compounds with 2+ unflagged votes (1,580)      0.773 ± 0.040   
fixed message passing, depth 2 (300)     all compounds (9,946)                          0.839 ± 0.009   
                                         compounds with 2+ unflagged votes (1,580)      0.815 ± 0.034   
graph, radius 0 (51)                     all compounds (9,946)                          0.835 ± 0.016   
                                         compounds with 2+ unflagged votes (1,580)      0.791 ± 0.055   
fixed message passing, depth 3 (300)     all compounds (9,946)                          0.817 ± 0.009   
                                         compounds with 2+ unflagged votes (1,580)      0.792 ± 0.032   
ESOL (4)                                 all compounds (9,946)                          0.769 ± 0.010   
                                         compounds with 2+ unflagged votes (1,580)      0.764 ± 0.053   
repeat measurement (benchmark)           compounds with 2+ unflagged votes (1,580)              0.963   

                                                                                   share of tier entropy (%)  \
feature set                              subset                                                                
RDKit, all (187)                         all compounds (9,946)                                    51.8 ± 0.5   
                                         compounds with 2+ unflagged votes (1,580)                50.1 ± 2.1   
RDKit, one per group at |ρ| ≥ 0.95 (165) all compounds (9,946)                                    51.8 ± 0.5   
                                         compounds with 2+ unflagged votes (1,580)                50.0 ± 2.0   
RDKit, one per group at |ρ| ≥ 0.9 (152)  all compounds (9,946)                                    51.8 ± 0.4   
                                         co

In [8]:
# Fixed message passing: how much do the scores move with the random weights? Mean over splits per seed, then range over seeds.
per_seed = results[(results["side"] == "validation") & results["feature set"].str.startswith("fixed message passing")]
per_seed = per_seed.groupby(["feature set", "subset"], sort=False)[list(METRICS)].mean().reset_index()
per_seed["depth"] = per_seed["feature set"].str.extract(r"depth (\d+)", expand=False).astype(int)
by_depth = per_seed.groupby(["depth", "subset"])[list(METRICS)]
seed_spread = by_depth.min().round(METRICS).astype(str) + "–" + by_depth.max().round(METRICS).astype(str)
lam_by_set = results[results["side"] == "validation"].groupby("feature set")["λ"].mean().round(3)
print(f"shrinkage λ chosen on validation, mean over splits: {lam_by_set.min()}–{lam_by_set.max()} across feature sets")
seed_spread

shrinkage λ chosen on validation, mean over splits: 0.005–0.075 across feature sets


information (bits)  \
depth subset                                                         
1     all compounds (9,946)                            0.862–0.885   
      compounds with 2+ unflagged votes (1,580)        0.837–0.865   
2     all compounds (9,946)                            0.834–0.844   
      compounds with 2+ unflagged votes (1,580)        0.811–0.818   
3     all compounds (9,946)                            0.804–0.826   
      compounds with 2+ unflagged votes (1,580)        0.778–0.801   

                                                share of tier entropy (%)  \
depth subset                                                                
1     all compounds (9,946)                                     44.2–45.4   
      compounds with 2+ unflagged votes (1,580)                 42.2–43.6   
2     all compounds (9,946)                                     42.8–43.3   
      compounds with 2+ unflagged votes (1,580)                 40.9–41.3   
3     all compounds (9,946)                                     41.3–42.4   
      compounds with 2+ unflagged votes (1,580)                 39.2–40.4   

                                                accuracy: same tier (%)  \
depth subset                                                              
1     all compounds (9,946)                                   69.0–70.1   
      compounds with 2+ unflagged votes (1,580)               66.6–68.3   
2     all compounds (9,946)                                   68.6–68.7   
      compounds with 2+ unflagged votes (1,580)               66.5–67.1   
3     all compounds (9,946)                                   67.1–68.4   
      compounds with 2+ unflagged votes (1,580)               65.3–66.0   

                                                1 tier off (%)  \
depth subset                                                     
1     all compounds (9,946)                          26.2–27.0   
      compounds with 2+ unflagged votes (1,580)      26.8–28.0   
2     all compounds (9,946)                          27.0–27.2   
      compounds with 2+ unflagged votes (1,580)      27.3–27.8   
3     all compounds (9,946)                          27.4–28.4   
      compounds with 2+ unflagged votes (1,580)      28.4–28.9   

                                                2+ tiers off (%)  
depth subset                                                      
1     all compounds (9,946)                              3.7–4.0  
      compounds with 2+ unflagged votes (1,580)          4.9–5.4  
2     all compounds (9,946)                              4.2–4.3  
      compounds with 2+ unflagged votes (1,580)          5.1–5.7  
3     all compounds (9,946)                              4.2–4.6  
      compounds with 2+ unflagged votes (1,580)          5.6–5.8

### Reading the validation table

- **The RDKit descriptor sets lead**, at 1.006–1.009 bits over all compounds (about 52% of the tier's entropy), and differ from each other by less than 0.003 bits: dropping correlated descriptors, down to one per group at $|\rho| \ge 0.8$ (131 of 187), costs nothing the data can detect.
- **Fewer, hand-picked descriptors give the forest less to work with.** AqSolDB's 17 reach 0.947 bits and ESOL's 4 reach 0.769.
- **The graph representations sit in between, at 0.82–0.90 bits**, below AqSolDB's 17 despite having hundreds to thousands of features. Looking further out in the graph gives the forest less, not more: environments peak at radius 1 (0.899) and fall to 0.846 at radius 3, and fixed message passing falls from 0.871 at depth 1 to 0.817 at depth 3. Read this as a statement about the estimator, not the representation: a radius-3 environment set contains everything the radius-1 set does, so it cannot hold less information — an untuned forest spreading its splits over 4,069 sparse columns simply extracts less of it. A learned model of the graph could reverse the order; this forest cannot.
- **The message-passing seeds agree.** Their scores move by about 0.02 bits between seeds, so the pooled rows are not the luck of one draw of weights.
- **Calibration mattered little once $\lambda$ had a floor**: the shrinkage chosen on validation stayed between 0.005 and 0.075, so the forests' probabilities were nearly usable as they came.
- **On the benchmark population the descriptor forests match a second measurement on information** — 0.99 bits against 0.963 — **while agreeing with the measured tier less often** (71% against 76.5%). That is not a contradiction: a probability spread over the plausible tiers carries more information than one noisy draw that commits to a single tier, and the information score credits it. It is also what the framework's caution predicted: the repeat benchmark is a floor on what a good model can reach, not a ceiling. The audit below checks that this survives on folds the forests were not calibrated on.

Several sets are close, so the next step chooses between them with the standard error in mind, and with fewer features preferred.

## Choosing feature sets

We want the most information from the fewest features. We use the one-standard-error rule (Breiman et al. 1984; Hastie, Tibshirani & Friedman, *Elements of Statistical Learning*, §7.10). The score is validation information over all compounds. Every feature set is scored on the same 10 splits, so each is compared with the best through its per-split difference from it (paired), with the standard error corrected for overlapping training sets. Sets whose mean difference is within one standard error of zero are tied with the best: the data cannot tell them apart. The five seeds of each fixed message-passing depth are averaged per split first, so a lucky draw of weights is not rewarded. One standard error is a convention, so the threshold is our choice.

Being tied with the best only finds the smallest set at the top of the range. To see what fewer features cost, we also trace the **information–size front**: going from the fewest features to the most, a set joins the front only if it beats every set with fewer features by more than one standard error of their paired difference. Each step along the front is then a clear gain in information paid for with more features; a set off the front has a smaller set that does as well.

Three things follow, all decided here on validation alone:

- **Inspected** for feature importance: every set on the front or tied with the best, except fixed message passing, whose features are not individually readable.
- **Scored on the test folds** as an audit: every set on the front, the simplest set tied with the best (fewest features), and the best set. Choosing sets on validation makes their validation scores optimistic, which is why the chosen sets are scored once more on folds not used to choose them. The audit also runs on the cluster splits.
- **Carried downstream**: the last step on the front — the largest set that still clearly beats every smaller one. That rule is fixed now, before the test scores are seen.

In [9]:
SELECT_METRIC, SELECT_SUBSET = "information (bits)", ALL
scores = results[(results["side"] == "validation") & (results["subset"] == SELECT_SUBSET)]
by_split = scores.groupby([scores["feature set"].map(family), "split"])[SELECT_METRIC].mean().unstack()  # seeds averaged
best = by_split.mean(axis=1).idxmax()
difference = by_split.sub(by_split.loc[best], axis=1)  # per split, paired with the best
CORRECTION = 1 / len(CV_SPLITS) + test_to_train(CV_SPLITS)
selection = pd.DataFrame(
    {
        SELECT_METRIC: by_split.mean(axis=1),
        "difference from best": difference.mean(axis=1),
        "corrected SE of difference": np.sqrt(CORRECTION * difference.var(axis=1)),
        "features": [FEATURE_SETS[members(fam)[0]].shape[1] for fam in by_split.index],
    },
    index=by_split.index,
)
selection["tied with best"] = selection["difference from best"] >= -selection["corrected SE of difference"]
tied = selection[selection["tied with best"]]
simplest = tied.sort_values(["features", SELECT_METRIC], ascending=[True, False]).index[0]


def clearly_better(a: str, b: str) -> bool:
    """Whether `a` beats `b` on validation information by more than one corrected SE of their per-split difference."""
    d = by_split.loc[a] - by_split.loc[b]
    return d.mean() > np.sqrt(CORRECTION * d.var())


# Information vs feature count front: a set is kept only if it clearly beats every set with fewer features.
by_size = list(selection.sort_values(["features", SELECT_METRIC], ascending=[True, False]).index)
front = [fam for i, fam in enumerate(by_size) if all(clearly_better(fam, smaller) for smaller in by_size[:i])]
selection["on information–size front"] = selection.index.isin(front)

INSPECT = [name for fam in dict.fromkeys([*front, *tied.index]) if not fam.startswith("fixed message passing") for name in members(fam)]
CHOSEN = [name for fam in dict.fromkeys([*front, simplest, best]) for name in members(fam)]
DOWNSTREAM = members(front[-1])[0]  # the rule declared above: the last step on the front
print(f"best: {best}\nsimplest tied with the best: {simplest}\ninformation–size front: {front}\ncarried downstream: {DOWNSTREAM}")
selection.sort_values(SELECT_METRIC, ascending=False).round(3).rename_axis("feature set")

best: RDKit, all (187)
simplest tied with the best: RDKit, one per group at |ρ| ≥ 0.9 (152)
information–size front: ['ESOL (4)', 'AqSolDB (17)', 'RDKit, one per group at |ρ| ≥ 0.8 (131)', 'RDKit, one per group at |ρ| ≥ 0.9 (152)']
carried downstream: RDKit, one per group at |ρ| ≥ 0.9 (152)


,information (bits),difference from best,corrected SE of difference,features,tied with best,on information–size front
feature set,,,,,,
"RDKit, all (187)",1.009,0.000,0.000,187,True,False
"RDKit, one per group at |ρ| ≥ 0.95 (165)",1.009,-0.000,0.002,165,True,False
"RDKit, one per group at |ρ| ≥ 0.9 (152)",1.009,-0.000,0.003,152,True,True
"RDKit, one per group at |ρ| ≥ 0.8 (131)",1.006,-0.003,0.003,131,False,True
AqSolDB (17),0.947,-0.062,0.006,17,False,True
"graph, radius 1 (494)",0.899,-0.110,0.010,494,False,False
atom + bond feature sums (145),0.877,-0.132,0.009,145,False,False
"fixed message passing, depth 1 (300)",0.871,-0.139,0.006,300,False,False
"graph, radius 2 (2107)",0.867,-0.142,0.013,2107,False,False


The front has four steps: ESOL (4 features, 0.769 bits), AqSolDB (17 features, 0.947 bits), RDKit at $|\rho| \ge 0.8$ (131 features, 1.006 bits) and RDKit at $|\rho| \ge 0.9$ (152 features, 1.009 bits). The last step is a narrow one: the 152 beat the 131 by 0.003 bits, and the paired standard error of that difference is of the same size, so the set joins the front by the letter of the rule, not by anything the data show clearly — a reader who preferred the 131 could not be contradicted. The best set is the full 187, tied with the 165 and the 152; the simplest set tied with the best is $|\rho| \ge 0.9$ (152). By the rule fixed above, that 152-feature set is carried downstream: the largest set that still clearly beats every smaller one. The audit below shows what the choice between 131 and 152 is worth on folds that played no part in it. Before the audit, we look at which features carry their information.

## Which features carry the information

Permutation importance on the validation folds: shuffle features across compounds (not rows, so each compound still counts once) and measure the drop in information, with the split's calibration $\lambda$ held fixed. We use this rather than the forest's built-in importance, which is measured on the training rows and favours continuous features. Neither handles correlated features on its own: shuffle one, and the forest falls back on a correlated partner, so both look unimportant. So we shuffle correlated features together. Within each feature set, features linked by $|\rho_\text{Spearman}| \ge 0.8$, directly or through a chain of such pairs, form one group, scored as one; no feature outside a group reaches that level with one inside it. The level is our choice, and chaining can make groups large, so the table gives each group's size. Fixed message-passing features are random projections, so their importance is not individually readable. The feature sets inspected are those on the front or tied with the best (`INSPECT`, above).

Each table is headed "Feature importance: *feature set*", so which feature set it describes is explicit; the line under it gives that feature set's unshuffled validation information and how many of its groups clear the significance bar.

Reading each table, one row per group:

- **row label**: the group's name, its best-known member (AqSolDB's descriptors first), plus how many correlated features it holds, so the same group carries the same name across feature sets.
- **information lost when group shuffled (bits)**: how much validation information drops when that group is shuffled, mean over the 10 splits ± corrected standard error.
- **share of unshuffled information (%)**: that loss as a percentage of the model's information before any shuffling.
- **group size (features)**: how many individual feature columns are bundled into the group.
- **group members**: in a reduced RDKit set each descriptor is expanded to the correlated descriptors it stands for; an environment is a substructure; a feature sum is a count of one atom or bond property.

Only groups whose loss is clearly above zero (more than two standard errors, our choice) are listed, at most 20; the heading says how many there are. Losses do not add up to the model's information: groups overlap in what they say about solubility, so the forest can partly make up for a shuffled group with the others.

In [10]:
from IPython.display import Markdown

REPEATS = 5
SHUFFLE_LEVEL = 0.8  # features linked by |Spearman rho| >= this, directly or through a chain, are shuffled together; our choice
CLEAR = 2  # a group's loss counts as clearly above zero beyond this many standard errors; our choice
TOP = 20  # most groups listed per table


def readable(name: str, features: list[str]) -> list[str]:
    """What shuffled features stand for: descriptors (expanded to their correlation groups), substructures or names."""
    return [d for f in features for d in GROUPS.get(name, {}).get(f, [ENVIRONMENTS.get(f, f)])]


def permutation_importance(name: str, repeats: int = REPEATS, level: float = SHUFFLE_LEVEL) -> pd.DataFrame:
    """Information lost on the validation rows when each group of correlated features is shuffled across compounds."""
    groups = correlation_groups(FEATURE_SETS[name], level, linkage="single")
    rng = np.random.default_rng(0)
    X_all = FEATURE_SETS[name].reindex(values["InChIKey"]).reset_index(drop=True)
    y, w = values["tier"].to_numpy(), values["weight"].to_numpy()
    rows, unshuffled = [], []
    for split in CV_SPLITS:
        where = values["InChIKey"].map(splits[split]).to_numpy()
        train, held = where == "train", where == "validation"
        forest = RandomForestClassifier(**RF_SETTINGS).fit(X_all[train], y[train], sample_weight=w[train])
        X = X_all[held].reset_index(drop=True)
        compound = pd.factorize(values["InChIKey"].to_numpy()[held])[0]  # row -> compound
        first = np.unique(compound, return_index=True)[1]  # one row per compound
        proba = forest.predict_proba(X)
        lam = calibrate(y[held], proba, w[held])
        base = information(y[held], proba, w[held], lam)[0]
        unshuffled.append(base)
        for group, cols in groups.items():
            for _ in range(repeats):
                shuffled = X.copy()
                shuffled[cols] = X[cols].to_numpy()[first][rng.permutation(len(first))][compound]  # shuffle compounds
                rows.append({"split": split, "group": group, "loss": base - information(y[held], forest.predict_proba(shuffled), w[held], lam)[0]})
    loss = pd.DataFrame(rows).groupby(["group", "split"])["loss"].mean().unstack()  # group x split
    importance = pd.DataFrame(
        {
            "information lost (bits)": loss.mean(axis=1),
            "SE": np.sqrt(CORRECTION * loss.var(axis=1)),
            "share of unshuffled information (%)": 100 * loss.mean(axis=1) / np.mean(unshuffled),
            "contains": [readable(name, groups[g]) for g in loss.index],
        },
        index=loss.index,
    ).sort_values("information lost (bits)", ascending=False)
    importance.attrs["unshuffled"] = np.mean(unshuffled)
    return importance


def group_label(contains: list[str]) -> str:
    """A group's name: its best-known member (AqSolDB's descriptors first), plus how many others it holds."""
    head = next((d for d in AQSOLDB_17 if d in contains), contains[0])
    return head if len(contains) == 1 else f"{head} + {len(contains) - 1} correlated"


def show_importance(name: str) -> None:
    importance = permutation_importance(name)
    clear = importance[importance["information lost (bits)"] > CLEAR * importance["SE"]]
    shown = clear.head(TOP)
    listed = f"the top {TOP} are listed" if len(clear) > TOP else "all are listed"
    display(Markdown(
        f"#### Feature importance: {name}\n\nUnshuffled validation information: {importance.attrs['unshuffled']:.3f} bits. "
        f"{len(clear)} of {len(importance)} groups lose information clearly above zero (more than {CLEAR} standard errors); {listed}."
    ))
    table = pd.DataFrame(
        {
            "information lost when group shuffled (bits)": [f"{m:.3f} ± {se:.3f}" for m, se in zip(shown["information lost (bits)"], shown["SE"])],
            "share of unshuffled information (%)": [f"{v:.1f}" for v in shown["share of unshuffled information (%)"]],
            "group size (features)": [len(c) for c in shown["contains"]],
            "group members": ["; ".join(c[:8]) + (f"; … (+{len(c) - 8} more)" if len(c) > 8 else "") for c in shown["contains"]],
        }
    )
    with pd.option_context("display.max_colwidth", None):
        display(table.set_axis([group_label(c) for c in shown["contains"]]).rename_axis("feature group"))


for name in INSPECT:
    show_importance(name)

#### Feature importance: ESOL (4)

Unshuffled validation information: 0.769 bits. 4 of 4 groups lose information clearly above zero (more than 2 standard errors); all are listed.

,information lost when group shuffled (bits),share of unshuffled information (%),group size (features),group members
feature group,,,,
MolLogP,1.314 ± 0.039,170.8,1,MolLogP
MolWt,0.357 ± 0.010,46.4,1,MolWt
AromaticProportion,0.101 ± 0.012,13.1,1,AromaticProportion
NumRotatableBonds,0.072 ± 0.009,9.3,1,NumRotatableBonds


#### Feature importance: AqSolDB (17)

Unshuffled validation information: 0.947 bits. 9 of 9 groups lose information clearly above zero (more than 2 standard errors); all are listed.

,information lost when group shuffled (bits),share of unshuffled information (%),group size (features),group members
feature group,,,,
MolLogP,0.619 ± 0.012,65.3,1,MolLogP
MolWt + 6 correlated,0.616 ± 0.019,65.1,7,MolWt; MolMR; HeavyAtomCount; NumValenceElectrons; RingCount; LabuteASA; BertzCT
NumHAcceptors + 1 correlated,0.150 ± 0.005,15.9,2,NumHAcceptors; TPSA
BalabanJ,0.064 ± 0.005,6.7,1,BalabanJ
NumHeteroatoms,0.031 ± 0.003,3.3,1,NumHeteroatoms
NumHDonors,0.028 ± 0.003,2.9,1,NumHDonors
NumRotatableBonds,0.014 ± 0.002,1.4,1,NumRotatableBonds
NumAromaticRings,0.008 ± 0.003,0.9,1,NumAromaticRings
NumSaturatedRings + 1 correlated,0.006 ± 0.001,0.7,2,NumSaturatedRings; NumAliphaticRings


#### Feature importance: RDKit, one per group at |ρ| ≥ 0.8 (131)

Unshuffled validation information: 1.006 bits. 102 of 127 groups lose information clearly above zero (more than 2 standard errors); the top 20 are listed.

,information lost when group shuffled (bits),share of unshuffled information (%),group size (features),group members
feature group,,,,
MolWt + 23 correlated,0.306 ± 0.008,30.4,24,MolWt; HeavyAtomMolWt; ExactMolWt; Chi0v; Kappa1; LabuteASA; Chi1v; Chi2v; … (+16 more)
MolLogP,0.246 ± 0.004,24.4,1,MolLogP
PEOE_VSA6,0.050 ± 0.003,4.9,1,PEOE_VSA6
NumAromaticRings + 5 correlated,0.034 ± 0.002,3.4,6,SMR_VSA7; SlogP_VSA6; VSA_EState6; NumAromaticCarbocycles; NumAromaticRings; fr_benzene
BCUT2D_LOGPHI,0.033 ± 0.002,3.3,1,BCUT2D_LOGPHI
HallKierAlpha,0.028 ± 0.002,2.8,1,HallKierAlpha
SlogP_VSA2,0.026 ± 0.001,2.6,1,SlogP_VSA2
FpDensityMorgan1 + 1 correlated,0.024 ± 0.001,2.3,2,FpDensityMorgan1; FpDensityMorgan2
qed,0.023 ± 0.001,2.3,1,qed


... (2 outputs left out)

#### Feature importance: RDKit, all (187)

Unshuffled validation information: 1.009 bits. 98 of 124 groups lose information clearly above zero (more than 2 standard errors); the top 20 are listed.

,information lost when group shuffled (bits),share of unshuffled information (%),group size (features),group members
feature group,,,,
MolWt + 26 correlated,0.676 ± 0.013,67.0,27,MolWt; HeavyAtomMolWt; ExactMolWt; NumValenceElectrons; AvgIpc; BertzCT; Chi0; Chi0n; … (+19 more)
MolLogP,0.164 ± 0.004,16.3,1,MolLogP
NumAromaticRings + 5 correlated,0.070 ± 0.003,6.9,6,SMR_VSA7; SlogP_VSA6; VSA_EState6; NumAromaticCarbocycles; NumAromaticRings; fr_benzene
NumHAcceptors + 3 correlated,0.048 ± 0.003,4.8,4,TPSA; NOCount; NumHAcceptors; NumHeteroatoms
MinPartialCharge + 1 correlated,0.036 ± 0.002,3.6,2,MinPartialCharge; MaxAbsPartialCharge
SlogP_VSA2,0.028 ± 0.002,2.8,1,SlogP_VSA2
FpDensityMorgan1 + 2 correlated,0.027 ± 0.002,2.7,3,FpDensityMorgan1; FpDensityMorgan2; FpDensityMorgan3
PEOE_VSA6,0.027 ± 0.002,2.6,1,PEOE_VSA6
BCUT2D_MRHI + 1 correlated,0.018 ± 0.001,1.8,2,BCUT2D_MRHI; SMR_VSA10


#### Feature importance: RDKit, one per group at |ρ| ≥ 0.95 (165)

Unshuffled validation information: 1.009 bits. 101 of 125 groups lose information clearly above zero (more than 2 standard errors); the top 20 are listed.

,information lost when group shuffled (bits),share of unshuffled information (%),group size (features),group members
feature group,,,,
MolWt + 26 correlated,0.480 ± 0.009,47.5,27,MolWt; HeavyAtomMolWt; ExactMolWt; Chi0v; LabuteASA; Kappa1; Chi3v; Chi4v; … (+19 more)
MolLogP,0.185 ± 0.004,18.4,1,MolLogP
NumAromaticRings + 5 correlated,0.089 ± 0.004,8.8,6,NumAromaticCarbocycles; fr_benzene; VSA_EState6; SMR_VSA7; SlogP_VSA6; NumAromaticRings
MinPartialCharge + 1 correlated,0.038 ± 0.002,3.8,2,MinPartialCharge; MaxAbsPartialCharge
PEOE_VSA6,0.034 ± 0.002,3.4,1,PEOE_VSA6
FpDensityMorgan1 + 2 correlated,0.033 ± 0.002,3.3,3,FpDensityMorgan1; FpDensityMorgan2; FpDensityMorgan3
NumHAcceptors + 2 correlated,0.026 ± 0.002,2.6,3,TPSA; NOCount; NumHAcceptors
BCUT2D_LOGPHI,0.025 ± 0.002,2.5,1,BCUT2D_LOGPHI
BCUT2D_MRHI + 1 correlated,0.025 ± 0.002,2.4,2,BCUT2D_MRHI; SMR_VSA10


## Test folds: the audit

Scored once, for the feature sets chosen above (`CHOSEN`). The first table is the random cross-validation test side — folds that played no part in the choice; if choosing on validation had flattered the chosen sets, their test scores would come out noticeably lower. The second table is the same sets on the cluster splits, where a test compound has no close relative in training: how much of each representation's information survives unfamiliar chemistry.

In [11]:
summarize(results[results["feature set"].isin(CHOSEN)], "test", CV_SPLITS)

information (bits)  \
feature set                             subset                                                         
RDKit, one per group at |ρ| ≥ 0.9 (152) all compounds (9,946)                          1.009 ± 0.009   
                                        compounds with 2+ unflagged votes (1,580)      0.991 ± 0.041   
RDKit, all (187)                        all compounds (9,946)                          1.008 ± 0.009   
                                        compounds with 2+ unflagged votes (1,580)      0.990 ± 0.039   
RDKit, one per group at |ρ| ≥ 0.8 (131) all compounds (9,946)                          1.007 ± 0.009   
                                        compounds with 2+ unflagged votes (1,580)      0.992 ± 0.042   
AqSolDB (17)                            all compounds (9,946)                          0.946 ± 0.012   
                                        compounds with 2+ unflagged votes (1,580)      0.921 ± 0.042   
ESOL (4)                                all compounds (9,946)                          0.769 ± 0.011   
                                        compounds with 2+ unflagged votes (1,580)      0.753 ± 0.061   
repeat measurement (benchmark)          compounds with 2+ unflagged votes (1,580)              0.963   

                                                                                  share of tier entropy (%)  \
feature set                             subset                                                                
RDKit, one per group at |ρ| ≥ 0.9 (152) all compounds (9,946)                                    51.8 ± 0.4   
                                        compounds with 2+ unflagged votes (1,580)                50.0 ± 2.1   
RDKit, all (187)                        all compounds (9,946)                                    51.7 ± 0.4   
                                        compounds with 2+ unflagged votes (1,580)                49.9 ± 2.0   
RDKit, one per group at |ρ| ≥ 0.8 (131) all compounds (9,946)                                    51.7 ± 0.4   
                                        compounds with 2+ unflagged votes (1,580)                50.0 ± 2.1   
AqSolDB (17)                            all compounds (9,946)                                    48.5 ± 0.6   
                                        compounds with 2+ unflagged votes (1,580)                46.4 ± 2.1   
ESOL (4)                                all compounds (9,946)                                    39.4 ± 0.5   
                                        compounds with 2+ unflagged votes (1,580)                38.0 ± 3.0   
repeat measurement (benchmark)          compounds with 2+ unflagged votes (1,580)                             

                                                                                  accuracy: same tier (%)  \
feature set                             subset                                                              
RDKit, one per group at |ρ| ≥ 0.9 (152) all compounds (9,946)                                  73.7 ± 0.5   
                                        compounds with 2+ unflagged votes (1,580)              72.1 ± 1.6   
RDKit, all (187)                        all compounds (9,946)                                  73.6 ± 0.5   
                                        compounds with 2+ unflagged votes (1,580)              71.9 ± 1.5   
RDKit, one per group at |ρ| ≥ 0.8 (131) all compounds (9,946)                                  73.9 ± 0.6   
                                        compounds with 2+ unflagged votes (1,580)              72.0 ± 1.8   
AqSolDB (17)                            all compounds (9,946)                                  71.6 ± 0.5   
                                        compounds with 2+ unflagged votes (1,580)              69.7 ± 1.1   
ESOL (4)                                all compounds (9,946)                                  66.9 ± 0.5   
                                        compounds with 2+ unflagged votes (1,580)     

In [12]:
cluster_results = pd.concat([run(name, CLUSTER_SPLITS) for name in CHOSEN], ignore_index=True)
summarize(cluster_results, "test", CLUSTER_SPLITS)

information (bits)  \
feature set                             subset                                                         
RDKit, all (187)                        all compounds (9,946)                          0.943 ± 0.024   
                                        compounds with 2+ unflagged votes (1,580)      0.946 ± 0.047   
RDKit, one per group at |ρ| ≥ 0.9 (152) all compounds (9,946)                          0.941 ± 0.025   
                                        compounds with 2+ unflagged votes (1,580)      0.944 ± 0.047   
RDKit, one per group at |ρ| ≥ 0.8 (131) all compounds (9,946)                          0.938 ± 0.024   
                                        compounds with 2+ unflagged votes (1,580)      0.942 ± 0.047   
AqSolDB (17)                            all compounds (9,946)                          0.892 ± 0.028   
                                        compounds with 2+ unflagged votes (1,580)      0.896 ± 0.057   
ESOL (4)                                all compounds (9,946)                          0.725 ± 0.032   
                                        compounds with 2+ unflagged votes (1,580)      0.734 ± 0.045   
repeat measurement (benchmark)          compounds with 2+ unflagged votes (1,580)              0.963   

                                                                                  share of tier entropy (%)  \
feature set                             subset                                                                
RDKit, all (187)                        all compounds (9,946)                                    48.5 ± 0.9   
                                        compounds with 2+ unflagged votes (1,580)                47.9 ± 2.4   
RDKit, one per group at |ρ| ≥ 0.9 (152) all compounds (9,946)                                    48.4 ± 0.9   
                                        compounds with 2+ unflagged votes (1,580)                47.8 ± 2.3   
RDKit, one per group at |ρ| ≥ 0.8 (131) all compounds (9,946)                                    48.3 ± 0.9   
                                        compounds with 2+ unflagged votes (1,580)                47.7 ± 2.3   
AqSolDB (17)                            all compounds (9,946)                                    45.9 ± 1.2   
                                        compounds with 2+ unflagged votes (1,580)                45.4 ± 2.9   
ESOL (4)                                all compounds (9,946)                                    37.3 ± 1.5   
                                        compounds with 2+ unflagged votes (1,580)                37.2 ± 2.2   
repeat measurement (benchmark)          compounds with 2+ unflagged votes (1,580)                             

                                                                                  accuracy: same tier (%)  \
feature set                             subset                                                              
RDKit, all (187)                        all compounds (9,946)                                  71.6 ± 0.7   
                                        compounds with 2+ unflagged votes (1,580)              71.1 ± 2.1   
RDKit, one per group at |ρ| ≥ 0.9 (152) all compounds (9,946)                                  71.9 ± 0.7   
                                        compounds with 2+ unflagged votes (1,580)              71.0 ± 1.8   
RDKit, one per group at |ρ| ≥ 0.8 (131) all compounds (9,946)                                  72.0 ± 0.6   
                                        compounds with 2+ unflagged votes (1,580)              71.1 ± 1.7   
AqSolDB (17)                            all compounds (9,946)                                  69.5 ± 0.8   
                                        compounds with 2+ unflagged votes (1,580)              68.1 ± 2.5   
ESOL (4)                                all compounds (9,946)                                  65.0 ± 0.9   
                                        compounds with 2+ unflagged votes (1,580)     

## Conclusion

MolLogP and molecular size do the heavy lifting. In every descriptor set the group holding MolWt (weight, atom counts, surface area and their correlates) and MolLogP are the two largest losses under shuffling; in ESOL, taking MolLogP away costs the forest more bits than it had — deprived of its main feature it predicts confidently and wrongly, which the information score charges and a hard-tier accuracy would not.

The audit holds. On the random test folds the chosen sets reproduce their validation scores to within 0.005 bits (RDKit sets 1.007–1.009, AqSolDB 0.946, ESOL 0.769), so choosing on validation flattered nothing. On the cluster splits — where a test compound has no close relative in training — every set loses about 0.04–0.07 bits (RDKit 0.938–0.943, AqSolDB 0.892, ESOL 0.725) and about two points of same-tier accuracy: a real but modest cost of unfamiliar chemistry, and the order of the representations is unchanged. The set carried forward is RDKit at $|\rho| \ge 0.9$ (152 features), chosen on validation before any test score was seen; the audit confirms it does as well as the full 187, and that the 131-feature set it edged out on validation is 0.002 bits behind it on test — the choice between those two was never going to matter, and the rule only had to make it.

Against the benchmark. On compounds with 2+ unflagged votes, the 152-descriptor forest shares 0.99 bits with the measured tier on the random test folds and 0.94 on the cluster splits, against the 0.963 bits a second measurement shares: on information it is as good as measuring the compound again on familiar chemistry, and slightly short of that on unfamiliar chemistry. On the hard tier it is not — it lands in the measured tier 72% of the time against the repeat's 76.5%. The two readings differ because a hedged probability is worth more than a committed guess, and the information score is the one the tier design was built on. What remains between this and the reachable ceiling cannot be charged to the representation alone: it bundles what the descriptors leave out, what an untuned forest fails to extract, and the calibration's residual. The next notebooks probe which: [05_learning_to_split.ipynb](05_learning_to_split.ipynb) asks where the model fails on the hardest split of the same compounds, and [06_knn.ipynb](06_knn.ipynb) asks whether those failures are the representation's or the model's. The cell below writes the chosen feature set for them.

In [13]:
# The feature set carried forward, one row per compound, read by the later notebooks.
FEATURE_SETS[DOWNSTREAM].to_csv(ROOT / "data" / "aqsoldb_features.csv")
print(f"{DOWNSTREAM}: {len(FEATURE_SETS[DOWNSTREAM])} compounds -> data/aqsoldb_features.csv")

RDKit, one per group at |ρ| ≥ 0.9 (152): 9946 compounds -> data/aqsoldb_features.csv
